In [1]:
from google.cloud import secretmanager
import pandas as pd
from currensee.utils.db_utils import create_pg_engine
from sqlalchemy import text

In [2]:
from currensee.core.settings import Settings
settings = Settings()

In [3]:
import inspect
print(inspect.signature(create_pg_engine))

(db_name: str)


In [4]:
crm_outlook_engine = create_pg_engine('crm_outlook')

# Find missing accounts to align

In [5]:
query= ''' select * from email_data where (to_emails = 'jessica.palmer@hasbro.com' or from_email = 'jessica.palmer@hasbro.com')'''
email_df = pd.read_sql(query, crm_outlook_engine)

In [8]:
query= ''' select distinct to_emails from email_data '''
email_df = pd.read_sql(query, crm_outlook_engine)
email_df.head(20)

,to_emails
0,lisa.kennedy@lockheedmartincorporation.com
1,cynthia.hobbs@abbvie.com
2,kelly.smith@presidiopropertytrust.com
3,denise.moore@celestica.com
4,jane.moneypenny1@outlook.com
5,anna.lawrence@matson.com
6,tracey.smith@medtronic.com
7,michelle.jenkins@intuitivesurgical.com
8,timothy.ochoa@hyatthotels.com
9,jessica.palmer@hasbro.com


In [9]:
query= ''' select * from client_alignment where employee_last_name = 'Moneypenny' '''
df = pd.read_sql(query, crm_outlook_engine)
df.head(20)

,employee_id,employee_first_name,employee_last_name,account_id,company,industry,contact_first_name,contact_last_name,contact_email,contact_title,contact_phone
0,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,d02ccad1-26c4-4768-95a4-833dbc2b2460,Mariott,Hospitality,Mary,Vasquez,mary.vasquez@mariott.com,Senior Director,520-618-5303x572
1,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,d02ccad1-26c4-4768-95a4-833dbc2b2461,AbbVie,Healthcare,Cynthia,Hobbs,cynthia.hobbs@abbvie.com,Director,(446)673-8121x90878
2,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,d02ccad1-26c4-4768-95a4-833dbc2b2462,AeroVironment,Manufacturing,Jennifer,Phelps,jennifer.phelps@aerovironment.com,Senior Director,797-584-0061x89137
3,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,c42e62e8-e943-47e0-bfd1-651e053c8bc6,Amedisys,Healthcare,Kyle,Waters,kyle.waters@amedisys.com,Senior Director,590.239.3215x8014
4,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,c42e62e8-e943-47e0-bfd1-651e053c8bc5,Celestica,Technology,Denise,Moore,denise.moore@celestica.com,Senior Director,754-579-1511x763
5,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,c2352f32-9a89-4f6e-995e-3f1d61da1ef4,Compass,RealEstate,Adam,Clay,adam.clay@compass.com,VP,-2931
6,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,68c2a459-fc28-4f39-bf27-8721424c6db8,Guardant Health,Healthcare,Roberto,Martin,roberto.martin@guardanthealth.com,VP,420.200.4573x07741
7,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,fca6b8a7-e1f3-4a6d-918b-02d0b2b1c2b0,Lockheed Martin Corporation,Manufacturing,Lisa,Kennedy,lisa.kennedy@lockheedmartincorporation.com,Senior Director,+1-341-221-9798x577
8,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,962f2d76-c51c-4a8b-80c3-ecdb5db59858,Intuitive Surgical,Healthcare,Michelle,Jenkins,michelle.jenkins@intuitivesurgical.com,Director,(612)342-3657x6255
9,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,ed22214f-9fbe-4a2d-bf51-46d05f2bba98,Medtronic,Healthcare,Tracey,Smith,tracey.smith@medtronic.com,Director,+1-952-819-1211x857


In [16]:
query= ''' select * from(select distinct 
 account_id 
, company
, industry
, contact_first_name	
, contact_last_name	
, contact_email	
, contact_title	
, contact_phone
, max(case when employee_last_name = 'Moneypenny' then 1 else 0 end) as has_jane
from client_alignment
where contact_email in (select distinct to_emails from email_data where from_email = 'jane.moneypenny1@outlook.com')
group by 1,2,3,4,5,6,7,8
)a where has_jane =0
'''
email_df = pd.read_sql(query, crm_outlook_engine)
email_df

,account_id,company,industry,contact_first_name,contact_last_name,contact_email,contact_title,contact_phone,has_jane
0,0d99abf4-faf2-4885-9c29-9086d944b973,Presidio Property Trust,RealEstate,Kelly,Smith,kelly.smith@presidiopropertytrust.com,Senior Director,001-671-821-6029,0
1,3b911a12-ff3a-4a93-9f44-6454381d76bc,Hasbro,Retail,Jessica,Palmer,jessica.palmer@hasbro.com,VP,909.878.8329x31984,0
2,7599e1bb-df47-44b6-b206-e6b1a0fe62av,Ladder Capital Corp,RealEstate,Ronnie,Gray,ronnie.gray@laddercapitalcorp.com,Director,001-750-645-5770x695,0
3,bcc303fb-b897-47ff-b1ae-02e6fdb34c0d,ManpowerGroup,Manufacturing,David,Moreno,david.moreno@manpowergroup.com,Director,001-945-497-7155x003,0
4,c4b9e3e8-2c82-49d4-93ca-050fd83ad6be,Matson,Manufacturing,Anna,Lawrence,anna.lawrence@matson.com,Director,+1-675-665-6673x605,0
5,d3021a76-bf2c-4f23-9b8c-45323d214920,GameStop Corp,Retail,Amy,Winters,amy.winters@gamestopcorp.com,Senior Director,381-842-2729x61450,0
6,f43fa598-e705-4908-98ab-e231f0287da9,Hyatt Hotels,Hospitality,Timothy,Ochoa,timothy.ochoa@hyatthotels.com,Manager,315-583-8080,0


# Load and add missing accounts

In [17]:
df_append = pd.read_excel("other_clients_align.xlsx")

In [18]:
df_append

,employee_id,employee_first_name,employee_last_name,account_id,company,industry,contact_first_name,contact_last_name,contact_email,contact_title,contact_phone
0,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,0d99abf4-faf2-4885-9c29-9086d944b973,Presidio Property Trust,RealEstate,Kelly,Smith,kelly.smith@presidiopropertytrust.com,Senior Director,001-671-821-6029
1,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,3b911a12-ff3a-4a93-9f44-6454381d76bc,Hasbro,Retail,Jessica,Palmer,jessica.palmer@hasbro.com,VP,909.878.8329x31984
2,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,7599e1bb-df47-44b6-b206-e6b1a0fe62av,Ladder Capital Corp,RealEstate,Ronnie,Gray,ronnie.gray@laddercapitalcorp.com,Director,001-750-645-5770x695
3,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,bcc303fb-b897-47ff-b1ae-02e6fdb34c0d,ManpowerGroup,Manufacturing,David,Moreno,david.moreno@manpowergroup.com,Director,001-945-497-7155x003
4,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,c4b9e3e8-2c82-49d4-93ca-050fd83ad6be,Matson,Manufacturing,Anna,Lawrence,anna.lawrence@matson.com,Director,+1-675-665-6673x605
5,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,d3021a76-bf2c-4f23-9b8c-45323d214920,GameStop Corp,Retail,Amy,Winters,amy.winters@gamestopcorp.com,Senior Director,381-842-2729x61450
6,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,f43fa598-e705-4908-98ab-e231f0287da9,Hyatt Hotels,Hospitality,Timothy,Ochoa,timothy.ochoa@hyatthotels.com,Manager,315-583-8080


In [19]:
df_append.to_sql(
    name="client_alignment", 
    con=crm_outlook_engine, 
    if_exists="append",  # append rows to existing table
    index=False          # do not write DataFrame index as a column
)

7

In [20]:
query= ''' select * from client_alignment where employee_last_name = 'Moneypenny' '''
df = pd.read_sql(query, crm_outlook_engine)
df.head(20)

,employee_id,employee_first_name,employee_last_name,account_id,company,industry,contact_first_name,contact_last_name,contact_email,contact_title,contact_phone
0,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,d02ccad1-26c4-4768-95a4-833dbc2b2460,Mariott,Hospitality,Mary,Vasquez,mary.vasquez@mariott.com,Senior Director,520-618-5303x572
1,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,d02ccad1-26c4-4768-95a4-833dbc2b2461,AbbVie,Healthcare,Cynthia,Hobbs,cynthia.hobbs@abbvie.com,Director,(446)673-8121x90878
2,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,d02ccad1-26c4-4768-95a4-833dbc2b2462,AeroVironment,Manufacturing,Jennifer,Phelps,jennifer.phelps@aerovironment.com,Senior Director,797-584-0061x89137
3,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,c42e62e8-e943-47e0-bfd1-651e053c8bc6,Amedisys,Healthcare,Kyle,Waters,kyle.waters@amedisys.com,Senior Director,590.239.3215x8014
4,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,c42e62e8-e943-47e0-bfd1-651e053c8bc5,Celestica,Technology,Denise,Moore,denise.moore@celestica.com,Senior Director,754-579-1511x763
5,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,c2352f32-9a89-4f6e-995e-3f1d61da1ef4,Compass,RealEstate,Adam,Clay,adam.clay@compass.com,VP,-2931
6,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,68c2a459-fc28-4f39-bf27-8721424c6db8,Guardant Health,Healthcare,Roberto,Martin,roberto.martin@guardanthealth.com,VP,420.200.4573x07741
7,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,fca6b8a7-e1f3-4a6d-918b-02d0b2b1c2b0,Lockheed Martin Corporation,Manufacturing,Lisa,Kennedy,lisa.kennedy@lockheedmartincorporation.com,Senior Director,+1-341-221-9798x577
8,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,962f2d76-c51c-4a8b-80c3-ecdb5db59858,Intuitive Surgical,Healthcare,Michelle,Jenkins,michelle.jenkins@intuitivesurgical.com,Director,(612)342-3657x6255
9,31e735f6-cfbe-4107-8aba-4ef45a90e9ae,Jane,Moneypenny,ed22214f-9fbe-4a2d-bf51-46d05f2bba98,Medtronic,Healthcare,Tracey,Smith,tracey.smith@medtronic.com,Director,+1-952-819-1211x857
